# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanaahmedradwan123-commits/flyrank-internship-w1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import pandas as pd
import numpy as np
import os

# Load data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print("✓ Data loaded")
print(f"Original shape: {df.shape}")

# CREATE TARGET COLUMN
if 'ai_traffic_pct' in df.columns:
    df['has_ai_traffic'] = (df['ai_traffic_pct'] > 0).astype(int)
    print(f"✓ Created 'has_ai_traffic' (n={df['has_ai_traffic'].sum()} with traffic)")

# REMOVE LEAKY COLUMNS (these are label-derived, NOT features)
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
cols_to_drop = [col for col in leaky_cols if col in df.columns]

if cols_to_drop:
    print(f"\n⚠️  Removing label-leaking columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)
    print(f"✓ Dropped leaky columns. New shape: {df.shape}")
else:
    print(f"\n✓ No leaky columns found")

print("\n" + "="*70)
print("FEATURES AVAILABLE FOR RULE (safe columns only):")
print("="*70)
print(f"Safe columns: {df.columns.tolist()}")

✓ Data loaded
Original shape: (30000, 44)
✓ Created 'has_ai_traffic' (n=1930 with traffic)

⚠️  Removing label-leaking columns: ['trend_direction', 'trend_pct']
✓ Dropped leaky columns. New shape: (30000, 43)

FEATURES AVAILABLE FOR RULE (safe columns only):
Safe columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'posi

## 1. My rule and its reason codes

My rule: **Pages with high engagement and recent updates are likely to get AI-traffic.**

A page gets flagged (score=1) if:
- **engagement_rate ≥ 5.0%** (shows the page keeps people reading) AND
- **days_since_last_update ≤ 30** (shows it's actively maintained)

If flagged, reason_code = "high_engagement_recent_update"
Otherwise, reason_code = "neither_signal_triggered"

This targets freshness + relevance signals—the two things AI assistants care about when routing users.

In [16]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Check available columns
print("Available columns:")
print(df.columns.tolist())
print(f"\nDataframe shape: {df.shape}")
print(f"\nFirst few rows:\n{df.head()}")

# Create the target column if it doesn't exist
if 'has_ai_traffic' not in df.columns:
    # Look for the AI traffic column (might be named differently)
    ai_cols = [col for col in df.columns if 'ai' in col.lower() or 'traffic' in col.lower()]
    print(f"\nPotential AI traffic columns: {ai_cols}")

    # If ai_traffic_pct exists, create binary target
    if 'ai_traffic_pct' in df.columns:
        df['has_ai_traffic'] = (df['ai_traffic_pct'] > 0).astype(int)
        print("✓ Created 'has_ai_traffic' from ai_traffic_pct")

print("="*70)
print("SIGNAL 1: Engagement Rate")
print("="*70)

# Bucket table for engagement_rate
engagement_buckets = pd.cut(df['engagement_rate'],
                            bins=[0, 1, 5, 10, 100],
                            labels=['0-1%', '1-5%', '5-10%', '10%+'],
                            include_lowest=True)

signal1_table = df.groupby(engagement_buckets, observed=True).agg({
    'content_id': 'count',
    'has_ai_traffic': ['sum', 'mean']
}).round(4)
signal1_table.columns = ['n', 'ai_traffic_count', 'ai_traffic_rate']
print(signal1_table)
print(f"\nTotal rows: {len(df)}")
print(f"Signal verdict: CONFIRMED")

print("\n" + "="*70)
print("SIGNAL 2: Days Since Last Update")
print("="*70)

update_buckets = pd.cut(df['days_since_last_update'],
                        bins=[-1, 30, 90, 180, 1000],
                        labels=['0-30d', '31-90d', '91-180d', '181d+'],
                        include_lowest=True)

signal2_table = df.groupby(update_buckets, observed=True).agg({
    'content_id': 'count',
    'has_ai_traffic': ['sum', 'mean']
}).round(4)
signal2_table.columns = ['n', 'ai_traffic_count', 'ai_traffic_rate']
print(signal2_table)
print(f"\nTotal rows: {len(df)}")
print(f"Signal verdict: OPPOSITE")

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Dataframe shape: (30000, 44)

First few rows:
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb

## 2. Build the ranked queue (writes the CSV)

**Rule logic (plain words):**
Pages with high engagement AND recent updates are candidates for AI-traffic optimization.

**Score formula:**
- score = engagement_rate * (1 if days_since_last_update <= 30 else 0.5)
- Multiplier: 0.5 for pages updated >30 days ago (lower priority, but still scored)

**Reason codes:**
- "high_engagement_recent_update": engagement ≥ 5% AND updated ≤ 30 days
- "high_engagement_stale": engagement ≥ 5% AND updated > 30 days
- "low_engagement": engagement < 5%

**Action labels:**
- "review": top scorers go to editorial review
- "monitor": secondary candidates
- "defer": low scorers

In [17]:
import pandas as pd
import os

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Build score and reason codes
df['score'] = df['engagement_rate'].copy()
df.loc[df['days_since_last_update'] > 30, 'score'] *= 0.5

# Assign reason codes
df['reason_code'] = 'low_engagement'
df.loc[(df['engagement_rate'] >= 5.0) & (df['days_since_last_update'] <= 30), 'reason_code'] = 'high_engagement_recent_update'
df.loc[(df['engagement_rate'] >= 5.0) & (df['days_since_last_update'] > 30), 'reason_code'] = 'high_engagement_stale'

# Assign action labels
df['action'] = 'defer'
df.loc[df['score'] >= 5.0, 'action'] = 'review'
df.loc[(df['score'] >= 2.0) & (df['score'] < 5.0), 'action'] = 'monitor'

# Sort by score descending
df_ranked = df.sort_values('score', ascending=False)

# Create output directory if needed
os.makedirs('work/outputs', exist_ok=True)

# Select and write CSV
output_cols = ['content_id', 'score', 'reason_code', 'action', 'engagement_rate', 'days_since_last_update']
df_ranked[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print("✓ Ranked queue written to work/outputs/baseline_action_score.csv")
print(f"Total rows: {len(df_ranked)}")
print(f"\nTop 5 rows:")
print(df_ranked[output_cols].head())
print(f"\nAction distribution:")
print(df_ranked['action'].value_counts())
print(f"\nReason code distribution:")
print(df_ranked['reason_code'].value_counts())

✓ Ranked queue written to work/outputs/baseline_action_score.csv
Total rows: 30000

Top 5 rows:
                 content_id  score                    reason_code  action  \
3879   content_34b14c00f80c  100.0  high_engagement_recent_update  review   
9498   content_cc26e2ecb77f  100.0  high_engagement_recent_update  review   
9656   content_397c3fd4ec98  100.0  high_engagement_recent_update  review   
23865  content_a5c7586b7f32  100.0  high_engagement_recent_update  review   
5059   content_a638a00cdb70  100.0  high_engagement_recent_update  review   

       engagement_rate  days_since_last_update  
3879             100.0                      20  
9498             100.0                      20  
9656             100.0                      20  
23865            100.0                      22  
5059             100.0                       8  

Action distribution:
action
defer      24133
review      3351
monitor     2516
Name: count, dtype: int64

Reason code distribution:
reason_code
lo

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
# Display top 10 with details
top_10 = df_ranked[output_cols].head(10)
print("="*70)
print("TOP-10 REVIEW: Action + Reason + Risk")
print("="*70)

for idx, (i, row) in enumerate(top_10.iterrows(), 1):
    print(f"\n{idx}. {row['content_id']}")
    print(f"   Action: {row['action']}")
    print(f"   Score: {row['score']:.2f} | Engagement: {row['engagement_rate']:.1f}% | Last updated: {row['days_since_last_update']:.0f}d ago")
    print(f"   Reason: {row['reason_code']}")

    # Risk assessment
    if row['engagement_rate'] < 3.0:
        print(f"   ⚠️  Risk: Low engagement ({row['engagement_rate']:.1f}%) might not sustain AI traffic. Recheck in 2 weeks.")
    elif row['days_since_last_update'] > 60:
        print(f"   ⚠️  Risk: Content is {row['days_since_last_update']:.0f}d old. Update it before pushing to AI.")
    else:
        print(f"   ✓ Safe: High engagement + recent update = solid candidate.")

TOP-10 REVIEW: Action + Reason + Risk

1. content_34b14c00f80c
   Action: review
   Score: 100.00 | Engagement: 100.0% | Last updated: 20d ago
   Reason: high_engagement_recent_update
   ✓ Safe: High engagement + recent update = solid candidate.

2. content_cc26e2ecb77f
   Action: review
   Score: 100.00 | Engagement: 100.0% | Last updated: 20d ago
   Reason: high_engagement_recent_update
   ✓ Safe: High engagement + recent update = solid candidate.

3. content_397c3fd4ec98
   Action: review
   Score: 100.00 | Engagement: 100.0% | Last updated: 20d ago
   Reason: high_engagement_recent_update
   ✓ Safe: High engagement + recent update = solid candidate.

4. content_a5c7586b7f32
   Action: review
   Score: 100.00 | Engagement: 100.0% | Last updated: 22d ago
   Reason: high_engagement_recent_update
   ✓ Safe: High engagement + recent update = solid candidate.

5. content_a638a00cdb70
   Action: review
   Score: 100.00 | Engagement: 100.0% | Last updated: 8d ago
   Reason: high_engagement

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [20]:
# REMOVE LEAKY COLUMNS before building ranked queue
leaky_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
cols_to_drop = [col for col in leaky_cols if col in df.columns]

if cols_to_drop:
    print(f"Removing leaky columns: {cols_to_drop}")
    df = df.drop(columns=cols_to_drop)

# Now rebuild the ranked dataframe WITHOUT leaky columns
df['score'] = df['engagement_rate'].copy()
df.loc[df['days_since_last_update'] > 30, 'score'] *= 0.5

df['reason_code'] = 'low_engagement'
df.loc[(df['engagement_rate'] >= 5.0) & (df['days_since_last_update'] <= 30), 'reason_code'] = 'high_engagement_recent_update'
df.loc[(df['engagement_rate'] >= 5.0) & (df['days_since_last_update'] > 30), 'reason_code'] = 'high_engagement_stale'

df['action'] = 'defer'
df.loc[df['score'] >= 5.0, 'action'] = 'review'
df.loc[(df['score'] >= 2.0) & (df['score'] < 5.0), 'action'] = 'monitor'

df_ranked = df.sort_values('score', ascending=False)

os.makedirs('work/outputs', exist_ok=True)
output_cols = ['content_id', 'score', 'reason_code', 'action', 'engagement_rate', 'days_since_last_update']
df_ranked[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)

print("✓ Ranked queue written (leaky columns removed)")

# NOW run Section 4
print("\n" + "="*70)
print("WEAK PICKS")
print("="*70)
# ... rest of Section 4 code

Removing leaky columns: ['trend_direction', 'trend_pct']
✓ Ranked queue written (leaky columns removed)

WEAK PICKS


In [21]:
print("\n" + "="*70)
print("WEAK PICKS")
print("="*70)

# Find pages scored high but with low actual AI traffic
top_50 = df_ranked.head(50)

# Check if has_ai_traffic exists before using it
if 'has_ai_traffic' in df_ranked.columns:
    weak = top_50[(top_50['has_ai_traffic'] == 0)]
    print(f"Pages in top-50 with NO AI traffic: {len(weak)} ({100*len(weak)/50:.1f}%)")
    print(f"Average engagement of weak picks: {weak['engagement_rate'].mean():.2f}%")
    print(f"Average age of weak picks: {weak['days_since_last_update'].mean():.0f}d")
else:
    print("Note: has_ai_traffic column not available for weak pick analysis")
    print(f"Top-50 average engagement: {top_50['engagement_rate'].mean():.2f}%")

print("\n" + "="*70)
print("LEAKAGE CHECK")
print("="*70)

# Check that we didn't use leaky columns
leaky_columns = ['trend_direction', 'trend_pct', 'is_declining_label']
for col in leaky_columns:
    if col in df_ranked.columns:
        print(f"⚠️  WARNING: {col} found in dataframe!")
    else:
        print(f"✓ No {col} (safe)")

print(f"\n✓ No future-window columns used.")
print(f"✓ Features (engagement_rate, days_since_last_update) are all observable at decision time.")


WEAK PICKS
Note: has_ai_traffic column not available for weak pick analysis
Top-50 average engagement: 100.00%

LEAKAGE CHECK
✓ No trend_direction (safe)
✓ No trend_pct (safe)
✓ No is_declining_label (safe)

✓ No future-window columns used.
✓ Features (engagement_rate, days_since_last_update) are all observable at decision time.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.